# Data Scraping and Storage

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
import time
import pandas as pd 
import os

In [2]:
def extract_movie_name(movie_string): 
  try:
    parts = movie_string.split(". ", 1)
    if len(parts) > 1:
      return parts[1]
    else:
      return movie_string
  except :#Exception as e: 
    return ""


In [7]:
def get_Movie_data(movie_Genre):
    
    movie_url="https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31";
    driver = webdriver.Chrome()  
    # driver.get(movie_url);

    driver.get('https://www.imdb.com/search/title/?title_type=feature&genres='+movie_Genre+'&release_date=2024-01-01,2024-12-31');
    
    driver.set_page_load_timeout(120)

    driver.maximize_window()
    time.sleep(3)

    is_buttonexist=True

    while is_buttonexist:
        try:
            btnsee_more=driver.find_element(By.XPATH,"//*[@id='__next']/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/div[2]/div/span/button/span/span")
            ActionChains(driver).move_to_element(btnsee_more).perform()
            btnsee_more.click()
            time.sleep(7) 
            is_buttonexist=True
        except :#Exception as e: 
            # print(e)
            is_buttonexist=False 

    movies_ul=driver.find_elements(By.XPATH,"//*[@id='__next']/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li")

    movie_Name_list=[] 
    movie_StoryLine_list=[]

    for movie_li in movies_ul:
        try:
            movie_Name=movie_li.find_element(By.XPATH,"./div/div/div/div[1]/div[2]/div[1]/a/h3")
            extracted_name = extract_movie_name(movie_Name.text)
            movie_Name_list.append(extracted_name) 
        except:
            movie_Name_list.append('') 

        try:
            movie_StoryLine=movie_li.find_element(By.XPATH,"./div/div/div/div[2]/div[1]/div[1]")
            movie_StoryLine_list.append(movie_StoryLine.text) 
        except:
            movie_StoryLine_list.append('') 


    dfmovie=pd.DataFrame({
        "MovieName":movie_Name_list,
        "StoryLine":movie_StoryLine_list
        })


    if os.path.exists('movie_data.csv'):
        dfmovie.to_csv('movie_data.csv', mode='a', header=False, index=False)
    else:
        dfmovie.to_csv('movie_data.csv', mode='w', header=True, index=False)


    # dfmovie.to_csv('movie_data.csv',index=False)
    driver.quit()

In [ ]:

genres=['adventure','animation','biography','crime','family','fantasy','game-show','history','music','musical','mystery','news','reality-tv','sci-fi','sport','talk-show','war','western','drama','romance','action','comedy','horror','thriller']

for i in genres:
    get_Movie_data(i.lower())


# Data Cleaning and Preprocessing

In [10]:
dfmovies = pd.read_csv(r"movie_data.csv")
dfmovies.head()

,MovieName,StoryLine
0,Mufasa: The Lion King,"Mufasa, a cub lost and alone, meets a sympathe..."
1,Twisters,"Kate Carter, a retired tornado-chaser and mete..."
2,Gladiator II,After his home is conquered by the tyrannical ...
3,Moana 2,After receiving an unexpected call from her wa...
4,Flow,"Cat is a solitary animal, but as its home is d..."


In [11]:
dfmovies.info()
dfmovies.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31998 entries, 0 to 31997
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MovieName  31998 non-null  object
 1   StoryLine  26368 non-null  object
dtypes: object(2)
memory usage: 500.1+ KB


,MovieName,StoryLine
count,31998,26368
unique,18637,14391
top,The Instigators,Follows two robbers who must go on the run wit...
freq,13,13


In [12]:
dfmovies.isnull().sum()

MovieName       0
StoryLine    5630
dtype: int64

In [13]:

dfmovies['StoryLine'] = dfmovies['StoryLine'].fillna("")
dfmovies.isnull().sum()

MovieName    0
StoryLine    0
dtype: int64

In [14]:
duplicate_rows = dfmovies[dfmovies.duplicated(keep=False)]
duplicate_rows

,MovieName,StoryLine
0,Mufasa: The Lion King,"Mufasa, a cub lost and alone, meets a sympathe..."
1,Twisters,"Kate Carter, a retired tornado-chaser and mete..."
2,Gladiator II,After his home is conquered by the tyrannical ...
3,Moana 2,After receiving an unexpected call from her wa...
4,Flow,"Cat is a solitary animal, but as its home is d..."
...,...,...
31993,Angel from Hell,A dark kingdom's prophecy hinges on a mission ...
31994,Aitokunrin,
31995,The Castastics 4: Brookhaven Movie,"David, now an adult have adopted a kid, BabyDa..."
31996,Dating A Narcissist,Angie finds out Christopher is not the man she...


In [15]:
dfUnique = dfmovies.drop_duplicates(keep=False)

duplicate_rows = dfUnique[dfUnique.duplicated(keep=False)]
duplicate_rows

,MovieName,StoryLine


In [18]:
dfUnique.info()
dfUnique.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 12395 entries, 113 to 30807
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MovieName  12395 non-null  object
 1   StoryLine  12395 non-null  object
dtypes: object(2)
memory usage: 290.5+ KB


,MovieName,StoryLine
count,12395,12395
unique,12309,8685
top,Under Pressure,
freq,4,3703


In [19]:
dfUnique.to_csv("movie_cleaned_data.csv", index=True)